# Eve-3-SABER-1B — Pretraining Notebook

**Model:** Eve-3-SABER-1B  
**Architecture:** Dense decoder-only transformer, 1.0B parameters  
**Tokenizer:** GPT-2 BPE, vocab 50,257  
**Training target:** 100B tokens (5× Chinchilla-optimal)

---

## Architecture Summary

| Dimension | Value |
|-----------|-------|
| `d_model` | 2048 |
| `n_layers` | 24 |
| `n_heads` | 16, `head_dim` = 128 |
| `d_ff` | 2855 (tuned for 1.0B) |
| `vocab_size` | 50,257 |
| `max_seq_len` | 2048 |
| Positional encoding | RoPE (theta=10000) |
| Normalization | RMSNorm (pre-norm, no bias) |
| LM head | Tied to token embedding |

---

## Novel Components

### 1. Slip-Anchors (All 24 layers)
A learnable codebook of `n_anchors=64` vectors in `d_anchor=128` space. Each token scores itself against the codebook and uses the blended result to additively bias K and V *after* RoPE. This creates soft semantic routing without breaking FlashAttention compatibility.

- Codebook: `(64, 128)` per layer
- K-bias and V-bias projected from `d_anchor` → `head_dim`
- ~0.3M extra params per layer, ~7.2M total

### 2. Experience Stream (All 24 layers)
A per-token, low-dimensional side-channel `d_exp=256` that flows layer-to-layer (not token-to-token). Each layer projects its post-attention hidden state into this stream and uses a prediction error as an auxiliary *curiosity loss* (`L_curiosity = 0.01 × mean_error`).

- Stop-gradient on the target is **critical** for stability
- Initialized to zeros at the start of each sequence
- ~0.66M extra params per layer, ~15.8M total

### 3. Resonant FFN (Even layers: 0, 2, 4, …, 22)
Standard SwiGLU is blended with a sinusoidal modulation `sin(W_freq @ x)` via a learned per-layer scalar `alpha` (initialized so `sigmoid(alpha_raw) = 0.95` → nearly full SwiGLU at init).

- `output = α × ffn + (1-α) × ffn × (1 + sin(W_freq @ x))`
- ~4.2M extra params per resonant layer, ~50.4M total across 12 even layers
- Odd layers use standard SwiGLU only

---

## Compute Targets

| Platform | ETA (100B tokens) | Cost |
|----------|--------------------|------|
| RunPod H100 PCIe (1×) | ~13.9 days | ~$666 |
| RunPod H100 SXM (1×) | ~11.1 days | ~$717 |
| RunPod 2× H100 SXM | ~5.8 days | ~$756 |
| Local 2× RTX 3090 (DDP) | ~60 days | ~$120 elec |

**Recommended:** RunPod H100 PCIe Community Cloud @ $1.99/hr — best cost-per-token, no session limits, persistent pods.

---

## Data Mix

**Pretraining (100B tokens):** 55% FineWeb-Edu · 35% DCLM-Baseline · 10% StarCoderData + OpenWebMath  
**Annealing (last 10B):** 50% DCLM · 20% FineMath-4+ · 20% Stack-Edu · 10% Wikipedia + Cosmopedia v2

---

## Notebook Structure

1. [Cell 1] This overview  
2. [Cell 2] Environment setup  
3. [Cell 3] Configuration  
4. [Cell 4] Model loading & param count  
5. [Cell 5] Tokenizer setup  
6. [Cell 6] Dataset loading & sequence packing  
7. [Cell 7] Training utilities (WandB, LR schedule, checkpointing)  
8. [Cell 8] Training loop (custom + HF Trainer options)  
9. [Cell 9] Evaluation & component analysis  
10. [Cell 10] Save & push to HuggingFace Hub  
11. [Cell 11] Annealing phase

---
## Cell 2 — Environment Setup

In [ ]:
# ============================================================
# ENVIRONMENT SETUP
# Run this once. On RunPod/Colab the container already has
# CUDA, so we just install Python packages.
# flash-attn requires a build step; use --no-build-isolation
# if you hit compile issues.
# ============================================================

import sys

# Install core dependencies
!{sys.executable} -m pip install -q \
    torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

!{sys.executable} -m pip install -q \
    transformers>=4.40.0 \
    datasets>=2.19.0 \
    tokenizers>=0.19.0 \
    accelerate>=0.29.0 \
    wandb>=0.17.0 \
    safetensors>=0.4.0 \
    huggingface_hub>=0.22.0 \
    tqdm \
    einops \
    scipy

# Flash Attention 2 — requires CUDA toolkit; skip on CPU-only environments
!{sys.executable} -m pip install -q flash-attn --no-build-isolation || \
    echo "WARNING: flash-attn install failed — will fall back to PyTorch SDPA"

print("\n✓ Dependencies installed.")

In [ ]:
# ============================================================
# IMPORTS
# ============================================================

import os
import sys
import json
import math
import time
import glob
import shutil
import random
import logging
import hashlib
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional, Dict, List, Tuple, Any
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, IterableDataset

import numpy as np
from tqdm.auto import tqdm

# HuggingFace ecosystem
from transformers import (
    AutoTokenizer,
    GPT2Tokenizer,
    PreTrainedTokenizerFast,
    get_cosine_schedule_with_warmup,
)
from datasets import load_dataset, IterableDataset as HFIterableDataset
from accelerate import Accelerator
from accelerate.utils import set_seed
from huggingface_hub import HfApi, create_repo
import wandb

# Flash Attention 2 check
try:
    from flash_attn import flash_attn_func
    from flash_attn.bert_padding import unpad_input, pad_input
    FLASH_ATTN_AVAILABLE = True
    print("✓ FlashAttention-2 available")
except ImportError:
    FLASH_ATTN_AVAILABLE = False
    print("⚠  FlashAttention-2 not available — using PyTorch SDPA (still fast on Ampere+)")

# Version check
print(f"\nPyTorch:        {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name}  {props.total_memory // 1024**3} GB VRAM")

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%H:%M:%S',
)
logger = logging.getLogger(__name__)

In [ ]:
# ============================================================
# GOOGLE DRIVE BACKUP (Colab)
# Mount Drive for rolling checkpoint backup. If the runtime
# crashes, disconnects, or times out, your checkpoints survive.
# ============================================================

import os

GDRIVE_BACKUP_DIR = None  # Set below if Drive is available

try:
    from google.colab import drive
    drive.mount('/content/drive')
    GDRIVE_BACKUP_DIR = Path('/content/drive/MyDrive/eve-3-saber-1b')
    GDRIVE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)
    print(f"✓ Google Drive mounted. Backups → {GDRIVE_BACKUP_DIR}")
except (ImportError, Exception) as e:
    print(f"⚠  Google Drive not available ({e}). Checkpoints will be local only.")
    print("   This is fine for RunPod/local, but risky on Colab.")

# Rolling backup config
GDRIVE_KEEP_LAST_N = 2   # Keep last 2 checkpoints on Drive


---
## Cell 3 — Configuration

All hyperparameters live here. Hardware detection auto-sets batch sizes and precision.  
Toggle `ENABLE_ANCHORS`, `ENABLE_EXPERIENCE`, `ENABLE_RESONANT` for ablations.

In [ ]:
# ============================================================
# HARDWARE DETECTION
# ============================================================

def detect_hardware():
    """Auto-detect GPU tier and return recommended settings."""
    if not torch.cuda.is_available():
        return "cpu", 0, False
    
    gpu_name = torch.cuda.get_device_name(0).lower()
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1024**3
    
    if "h100" in gpu_name:
        return "h100", vram_gb, True   # no GC needed
    elif "a100" in gpu_name:
        return "a100", vram_gb, True
    elif "3090" in gpu_name or "4090" in gpu_name:
        return "3090", vram_gb, True   # GC recommended
    elif "l4" in gpu_name or "t4" in gpu_name:
        return "l4_t4", vram_gb, True
    else:
        return "unknown", vram_gb, vram_gb < 32

GPU_TIER, GPU_VRAM_GB, _ = detect_hardware()
N_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f"Detected GPU tier: {GPU_TIER}, VRAM: {GPU_VRAM_GB:.1f} GB, GPUs: {N_GPUS}")

# ============================================================
# ── MODEL CONFIGURATION ─────────────────────────────────────
# ============================================================

MODEL_NAME   = "eve-3-saber-1b"
HF_REPO      = "anthonym21/Eve-3-SABER-1B"   # HuggingFace repository
MODEL_DIR    = Path("./eve-3-saber-1b")       # Local model files
CKPT_DIR     = Path("./checkpoints")
LOG_DIR      = Path("./logs")

for _d in [MODEL_DIR, CKPT_DIR, LOG_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

# Architecture
D_MODEL      = 2048
N_LAYERS     = 24
N_HEADS      = 16
HEAD_DIM     = D_MODEL // N_HEADS          # 128
D_FF         = 2855                        # Tuned via param_counter.py --tune-dff
VOCAB_SIZE   = 50257                       # GPT-2 tokenizer
MAX_SEQ_LEN  = 2048
ROPE_THETA   = 10000.0

# Novel component dimensions
D_EXP        = 256    # Experience stream dimension
N_ANCHORS    = 64     # Slip-anchor codebook size
D_ANCHOR     = 128    # Anchor bottleneck dimension

# Ablation toggles
ENABLE_ANCHORS     = True
ENABLE_EXPERIENCE  = True
ENABLE_RESONANT    = True
PREDICTABILITY_MODE = False  # GPT-5.2 Thinking conservative mode

# ── Predictability Mode overrides ──────────────────────────
if PREDICTABILITY_MODE:
    # Anchors nearly off; experience stream small updates;
    # resonant FFN only in last 8 layers
    ANCHOR_GATE_BIAS = -3.0
    EXPERIENCE_SCALE = 0.05
    RESONANT_LAYERS  = list(range(N_LAYERS - 8, N_LAYERS, 2))  # last 8 even
else:
    ANCHOR_GATE_BIAS = 0.0
    EXPERIENCE_SCALE = 1.0
    RESONANT_LAYERS  = list(range(0, N_LAYERS, 2))             # all even layers

# ============================================================
# ── TRAINING CONFIGURATION ──────────────────────────────────
# ============================================================

TOTAL_TOKENS          = 100_000_000_000   # 100B tokens
SEQ_LEN               = 2048
EFFECTIVE_BATCH_TOKENS = 500_000          # ~0.5M tokens per optimizer step

# Per-GPU micro-batch — auto-sized by GPU tier
_MICRO_BATCH_MAP = {
    "h100":   6,
    "a100":   2,
    "3090":   1,
    "l4_t4":  1,
    "unknown": 1,
    "cpu":    1,
}
MICRO_BATCH = _MICRO_BATCH_MAP.get(GPU_TIER, 1)

# Gradient accumulation to hit effective batch tokens
# tokens_per_step = MICRO_BATCH * N_GPUS (min 1) * GRAD_ACCUM * SEQ_LEN
_world_size = max(N_GPUS, 1)
GRAD_ACCUM = max(1, EFFECTIVE_BATCH_TOKENS // (MICRO_BATCH * _world_size * SEQ_LEN))

# Verify
_actual_batch_tokens = MICRO_BATCH * _world_size * GRAD_ACCUM * SEQ_LEN
print(f"\n── Batch Configuration ──")
print(f"  micro_batch:          {MICRO_BATCH}")
print(f"  world_size:           {_world_size}")
print(f"  grad_accum:           {GRAD_ACCUM}")
print(f"  effective batch (toks): {_actual_batch_tokens:,}")

TOTAL_STEPS   = TOTAL_TOKENS // _actual_batch_tokens
print(f"  total steps:          {TOTAL_STEPS:,}")
print(f"  total tokens:         {TOTAL_TOKENS:,}")

# Optimizer
LR            = 3e-4
LR_MIN        = 3e-5   # cosine floor = 10% of peak
BETAS         = (0.9, 0.95)
WEIGHT_DECAY  = 0.1
GRAD_CLIP     = 1.0
WARMUP_STEPS  = 2000

# Loss weights
CURIOSITY_WEIGHT = 0.01

# Mixed precision
DTYPE_STR   = "bf16"   # "bf16" | "fp16" | "fp32"
DTYPE_TORCH = torch.bfloat16

# Gradient checkpointing (recommended on 3090, optional elsewhere)
GRADIENT_CHECKPOINTING = (GPU_TIER in ("3090", "l4_t4", "unknown", "cpu"))
print(f"  gradient_checkpointing: {GRADIENT_CHECKPOINTING}")

# Checkpointing
CHECKPOINT_EVERY = 1000   # steps
KEEP_LAST_N      = 3      # delete older checkpoints

# Logging
LOG_EVERY      = 10       # steps
EVAL_EVERY     = 500      # steps
SEED           = 42

# WandB
WANDB_PROJECT  = "eve-3-saber-1b"
WANDB_ENTITY   = None   # set to your W&B username if needed
WANDB_RUN_NAME = f"{MODEL_NAME}-{'full' if ENABLE_ANCHORS and ENABLE_EXPERIENCE and ENABLE_RESONANT else 'ablation'}"

# ============================================================
# ── DATA CONFIGURATION ──────────────────────────────────────
# ============================================================

DATA_MIX_PRETRAIN = [
    # (hf_dataset_id, config_or_split, weight, streaming)
    ("HuggingFaceFW/fineweb-edu", "sample-350BT", 0.55, True),
    ("mlfoundations/dclm-baseline-1.0", "default",      0.35, True),
    ("bigcode/starcoderdata",            "python",       0.07, True),
    ("open-web-math/open-web-math",      "train",        0.03, True),
]

DATA_MIX_ANNEAL = [
    ("mlfoundations/dclm-baseline-1.0",  "default",     0.50, True),
    ("HuggingFaceTB/finemath",           "finemath-4+", 0.20, True),
    ("HuggingFaceTB/smollm-corpus",      "python-edu",  0.20, True),
    ("wikimedia/wikipedia",              "20231101.en", 0.05, True),
    ("HuggingFaceTB/cosmopedia",         "web_samples_v2", 0.05, True),
]

ANNEAL_START_FRACTION = 0.90   # start annealing at 90% of training
ANNEAL_TOKENS = int(TOTAL_TOKENS * (1 - ANNEAL_START_FRACTION))

print(f"\n── Data Configuration ──")
print(f"  Pretraining tokens:  {int(TOTAL_TOKENS * ANNEAL_START_FRACTION):,}")
print(f"  Annealing tokens:    {ANNEAL_TOKENS:,}")
print(f"\n  Pretraining mix:")
for ds_id, cfg, w, _ in DATA_MIX_PRETRAIN:
    print(f"    {w*100:.0f}%  {ds_id}/{cfg}")

# ============================================================
# ── PLATFORM-SPECIFIC OVERRIDES ─────────────────────────────
# ── (uncomment the block that matches your environment) ─────
# ============================================================

# --- RunPod H100 PCIe 80GB (single GPU) ---
# MICRO_BATCH = 4
# GRAD_ACCUM  = 61   # 4 * 1 * 61 * 2048 = 499,712 tokens ≈ 500K
# GRADIENT_CHECKPOINTING = False

# --- RunPod 2× H100 SXM 80GB (DDP via accelerate) ---
# MICRO_BATCH = 4
# GRAD_ACCUM  = 30   # 4 * 2 * 30 * 2048 = 491,520 tokens
# GRADIENT_CHECKPOINTING = False

# --- Local 2× RTX 3090 24GB (DDP via accelerate) ---
# MICRO_BATCH = 1
# GRAD_ACCUM  = 122  # 1 * 2 * 122 * 2048 = 499,712 tokens
# GRADIENT_CHECKPOINTING = True

# --- Google Colab Pro+ H100 80GB (single GPU) [RECOMMENDED] ---
# MICRO_BATCH = 6
# GRAD_ACCUM  = 40   # 6 * 1 * 40 * 2048 = 491,520 tokens ≈ 500K
# GRADIENT_CHECKPOINTING = False
# NOTE: H100 has bf16 TF32 tensor cores. torch.compile() gives
# significant speedup on H100 — see Cell 13 (Model Compilation).

# --- Google Colab A100 40GB (single GPU) ---
# MICRO_BATCH = 2
# GRAD_ACCUM  = 122  # 2 * 1 * 122 * 2048 = 499,712 tokens
# GRADIENT_CHECKPOINTING = False

set_seed(SEED)
print(f"\n✓ Configuration complete. Seed: {SEED}")

---
## Cell 4 — Model Architecture & Parameter Count

The full Eve-3-SABER-1B model is defined here inline.  
If `configuration_saber.py` and `modeling_saber.py` exist in `MODEL_DIR`, they are imported instead.

In [ ]:
# ============================================================
# ROTARY POSITIONAL EMBEDDING (RoPE)
# ============================================================

def precompute_freqs_cis(head_dim: int, max_seq_len: int, theta: float = 10000.0):
    """Precompute complex RoPE frequencies. Returns shape (max_seq_len, head_dim//2)."""
    freqs = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
    t = torch.arange(max_seq_len).float()
    freqs = torch.outer(t, freqs)          # (T, head_dim//2)
    return torch.polar(torch.ones_like(freqs), freqs)  # complex64


def apply_rope(x: torch.Tensor, freqs_cis: torch.Tensor) -> torch.Tensor:
    """Apply rotary embeddings to (B, T, n_heads, head_dim) tensor."""
    # x: (B, T, H, D), freqs_cis: (T, D//2) complex
    B, T, H, D = x.shape
    x_f = x.float().reshape(B, T, H, D // 2, 2)
    x_c = torch.view_as_complex(x_f)                     # (B, T, H, D//2) complex
    freqs = freqs_cis[:T].unsqueeze(0).unsqueeze(2)       # (1, T, 1, D//2)
    x_rotated = torch.view_as_real(x_c * freqs)          # (B, T, H, D//2, 2)
    return x_rotated.reshape(B, T, H, D).to(x.dtype)

In [ ]:
# ============================================================
# RMSNORM
# ============================================================

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        norm = x.float().pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return (x.float() * norm).to(x.dtype) * self.weight

In [ ]:
# ============================================================
# SLIP-ANCHOR MODULE
# Biases K and V after RoPE using a soft-codebook lookup.
# ============================================================

class SlipAnchorModule(nn.Module):
    """
    Computes anchor-based K and V bias addends for one attention layer.
    Shared across all heads in the layer (broadcast over head dimension).
    """

    def __init__(
        self,
        d_model: int = D_MODEL,
        n_anchors: int = N_ANCHORS,
        d_anchor: int = D_ANCHOR,
        head_dim: int = HEAD_DIM,
        gate_bias: float = ANCHOR_GATE_BIAS,
    ):
        super().__init__()
        self.gate_bias = gate_bias

        # Project hidden state to anchor space
        self.W_anchor_down = nn.Linear(d_model, d_anchor, bias=False)

        # Learnable codebook
        self.anchors = nn.Embedding(n_anchors, d_anchor)

        # Bias projections to key and value spaces
        self.U_k = nn.Linear(d_anchor, head_dim, bias=False)
        self.U_v = nn.Linear(d_anchor, head_dim, bias=False)

        # Small init for U_k, U_v so anchor bias starts near zero
        nn.init.normal_(self.U_k.weight, std=0.01)
        nn.init.normal_(self.U_v.weight, std=0.01)

    def forward(self, h: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Args:
            h: raw hidden state (B, T, d_model) — before QKV proj
        Returns:
            k_bias:     (B, T, head_dim)  — add to each K head
            v_bias:     (B, T, head_dim)  — add to each V head
            anchor_scores: (B, T, n_anchors) — for entropy logging
        """
        h_anch   = self.W_anchor_down(h)                               # (B, T, d_anchor)
        # scores: (B, T, n_anchors)
        raw_scores = h_anch @ self.anchors.weight.T + self.gate_bias
        anchor_scores = torch.softmax(raw_scores, dim=-1)

        # Weighted codebook retrieval
        anchor_ctx = anchor_scores @ self.anchors.weight             # (B, T, d_anchor)

        k_bias = self.U_k(anchor_ctx)                                # (B, T, head_dim)
        v_bias = self.U_v(anchor_ctx)                                # (B, T, head_dim)
        return k_bias, v_bias, anchor_scores

In [ ]:
# ============================================================
# EXPERIENCE STREAM MODULE
# Per architecture spec. Maintains per-token low-dim side state
# flowing layer-to-layer. Emits curiosity prediction error.
# ============================================================

class ExperienceStreamLayer(nn.Module):
    """
    One layer's experience stream update.
    Call order per layer:
      curiosity, new_state = exp_layer(h_post_attn, experience_state)
    """

    def __init__(
        self,
        d_model: int = D_MODEL,
        d_exp:   int = D_EXP,
        scale:   float = EXPERIENCE_SCALE,
    ):
        super().__init__()
        self.scale = scale

        self.W_s    = nn.Linear(d_model, d_exp, bias=False)  # summarise h → exp
        self.W_pred = nn.Linear(d_exp,   d_exp, bias=False)  # predict next summary
        self.W_e    = nn.Linear(d_exp,   d_exp, bias=False)  # update gate

        # Small init so experience stream starts quiet
        nn.init.normal_(self.W_s.weight,    std=0.01)
        nn.init.normal_(self.W_pred.weight, std=0.01)
        nn.init.normal_(self.W_e.weight,    std=0.01)

    def forward(
        self,
        h: torch.Tensor,                # (B, T, d_model) — post attention hidden
        experience: torch.Tensor,       # (B, T, d_exp)   — current stream state
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Returns:
            curiosity:       (B, T) scalar prediction error per token
            new_experience:  (B, T, d_exp) updated stream
        """
        # Summarise current hidden state
        s = self.W_s(h) * self.scale          # (B, T, d_exp)

        # CRITICAL: stop gradient on the target to prevent trivial collapse
        s_sg = s.detach()

        # Predict what this layer's summary should be from stream
        s_pred = self.W_pred(experience)      # (B, T, d_exp)

        # Curiosity = L2 prediction error (scalar per token)
        curiosity = (s_sg - s_pred).pow(2).mean(dim=-1)  # (B, T)

        # Update experience stream with SiLU gated residual
        delta = F.silu(self.W_e(s))
        new_experience = experience + delta

        return curiosity, new_experience

In [ ]:
# ============================================================
# SWIGLU FFN  (used on all layers)
# RESONANT FFN (even layers only — blends SwiGLU with sin mod)
# ============================================================

class SwiGLUFFN(nn.Module):
    def __init__(self, d_model: int = D_MODEL, d_ff: int = D_FF):
        super().__init__()
        self.W1 = nn.Linear(d_model, d_ff, bias=False)
        self.W3 = nn.Linear(d_model, d_ff, bias=False)
        self.W2 = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.W2(F.silu(self.W1(x)) * self.W3(x))


class ResonantFFN(nn.Module):
    """
    Resonant FFN for even layers.
    output = alpha * ffn_out + (1-alpha) * ffn_out * (1 + sin(W_freq @ x))
    alpha is a per-layer scalar, sigmoid(alpha_raw) initialised to 0.95.
    """

    def __init__(self, d_model: int = D_MODEL, d_ff: int = D_FF):
        super().__init__()
        self.swiglu = SwiGLUFFN(d_model, d_ff)

        # Frequency modulation: (d_model, d_model)
        self.W_freq = nn.Linear(d_model, d_model, bias=False)

        # alpha_raw init: sigmoid(3.0) ≈ 0.952  →  nearly pure SwiGLU at start
        self.alpha_raw = nn.Parameter(torch.tensor(3.0))

        # Small init on W_freq so modulation starts near zero
        nn.init.normal_(self.W_freq.weight, std=0.02)

    @property
    def alpha(self) -> torch.Tensor:
        return torch.sigmoid(self.alpha_raw)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        ffn_out = self.swiglu(x)                        # standard SwiGLU
        mod     = torch.sin(self.W_freq(x))             # sinusoidal modulation
        a       = self.alpha
        return a * ffn_out + (1 - a) * ffn_out * (1 + mod)

In [ ]:
# ============================================================
# ATTENTION  (with Slip-Anchors + RoPE + FlashAttn2 / SDPA)
# ============================================================

class SABERAttention(nn.Module):
    def __init__(
        self,
        d_model:   int   = D_MODEL,
        n_heads:   int   = N_HEADS,
        head_dim:  int   = HEAD_DIM,
        enable_anchors: bool = True,
    ):
        super().__init__()
        self.n_heads  = n_heads
        self.head_dim = head_dim
        self.scale    = head_dim ** -0.5
        self.enable_anchors = enable_anchors

        # Projections (no bias, as per spec)
        self.q_proj  = nn.Linear(d_model, d_model, bias=False)
        self.k_proj  = nn.Linear(d_model, d_model, bias=False)
        self.v_proj  = nn.Linear(d_model, d_model, bias=False)
        self.o_proj  = nn.Linear(d_model, d_model, bias=False)

        if enable_anchors:
            self.anchors = SlipAnchorModule(
                d_model=d_model,
                head_dim=head_dim,
            )

    def forward(
        self,
        x: torch.Tensor,              # (B, T, d_model)
        freqs_cis: torch.Tensor,      # (T, head_dim//2) complex
        attention_mask: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        B, T, _ = x.shape
        H, D    = self.n_heads, self.head_dim

        # Linear projections → reshape to (B, T, H, D)
        Q = self.q_proj(x).view(B, T, H, D)
        K = self.k_proj(x).view(B, T, H, D)
        V = self.v_proj(x).view(B, T, H, D)

        # Apply RoPE to Q and K BEFORE anchor modulation (per spec)
        Q = apply_rope(Q, freqs_cis)
        K = apply_rope(K, freqs_cis)

        # Slip-anchor K/V bias (applied AFTER RoPE)
        anchor_scores = None
        if self.enable_anchors:
            k_bias, v_bias, anchor_scores = self.anchors(x)  # (B, T, head_dim)
            # Broadcast bias over all heads
            K = K + k_bias.unsqueeze(2)   # (B, T, 1, D) + (B, T, H, D)
            V = V + v_bias.unsqueeze(2)

        # Transpose to (B, H, T, D) for SDPA
        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)

        # Attention (causal)
        if FLASH_ATTN_AVAILABLE:
            # FlashAttention-2 expects (B, T, H, D)
            Q_ = Q.transpose(1, 2).contiguous()
            K_ = K.transpose(1, 2).contiguous()
            V_ = V.transpose(1, 2).contiguous()
            attn_out = flash_attn_func(Q_, K_, V_, causal=True)
            attn_out = attn_out.reshape(B, T, H * D)
        else:
            # PyTorch 2.0+ SDPA with causal mask
            attn_out = F.scaled_dot_product_attention(
                Q, K, V,
                attn_mask=None,
                dropout_p=0.0,
                is_causal=True,
            )  # (B, H, T, D)
            attn_out = attn_out.transpose(1, 2).reshape(B, T, H * D)

        out = self.o_proj(attn_out)
        return out, anchor_scores

In [ ]:
# ============================================================
# TRANSFORMER LAYER
# ============================================================

class SABERLayer(nn.Module):
    def __init__(
        self,
        layer_idx:         int,
        enable_anchors:    bool = True,
        enable_experience: bool = True,
        resonant_layers:   List[int] = None,
    ):
        super().__init__()
        self.layer_idx         = layer_idx
        self.enable_experience = enable_experience

        resonant_layers = resonant_layers or RESONANT_LAYERS
        use_resonant    = ENABLE_RESONANT and (layer_idx in resonant_layers)

        self.norm1   = RMSNorm(D_MODEL)
        self.norm2   = RMSNorm(D_MODEL)
        self.attn    = SABERAttention(enable_anchors=enable_anchors)
        self.ffn     = ResonantFFN() if use_resonant else SwiGLUFFN()
        self.is_resonant = use_resonant

        if enable_experience:
            self.experience_layer = ExperienceStreamLayer()

    def forward(
        self,
        x: torch.Tensor,
        freqs_cis: torch.Tensor,
        experience: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor], Optional[torch.Tensor], Optional[torch.Tensor]]:
        """
        Returns:
            h:              updated hidden state (B, T, d_model)
            experience:     updated experience stream (B, T, d_exp) or None
            curiosity:      per-token prediction error (B, T) or None
            anchor_scores:  (B, T, n_anchors) or None
        """
        # Pre-norm attention
        attn_out, anchor_scores = self.attn(self.norm1(x), freqs_cis)
        x = x + attn_out

        # Experience stream update (post-attention)
        curiosity = None
        if self.enable_experience and experience is not None:
            curiosity, experience = self.experience_layer(x, experience)

        # Pre-norm FFN
        x = x + self.ffn(self.norm2(x))

        return x, experience, curiosity, anchor_scores

In [ ]:
# ============================================================
# SLIPCORE-1B MODEL
# ============================================================

class SABER1B(nn.Module):
    def __init__(
        self,
        vocab_size:        int  = VOCAB_SIZE,
        d_model:           int  = D_MODEL,
        n_layers:          int  = N_LAYERS,
        max_seq_len:       int  = MAX_SEQ_LEN,
        enable_anchors:    bool = ENABLE_ANCHORS,
        enable_experience: bool = ENABLE_EXPERIENCE,
        enable_resonant:   bool = ENABLE_RESONANT,
    ):
        super().__init__()
        self.d_model           = d_model
        self.n_layers          = n_layers
        self.enable_experience = enable_experience

        # Token embedding (LM head will be weight-tied)
        self.embed_tokens = nn.Embedding(vocab_size, d_model)
        self.embed_drop   = nn.Dropout(0.0)  # no dropout during pretraining

        # Transformer layers
        self.layers = nn.ModuleList([
            SABERLayer(
                layer_idx=i,
                enable_anchors=enable_anchors,
                enable_experience=enable_experience,
            )
            for i in range(n_layers)
        ])

        self.norm = RMSNorm(d_model)

        # LM head — weight-tied to embedding (saves ~103M params)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.embed_tokens.weight

        # Precompute RoPE frequencies
        self.register_buffer(
            "freqs_cis",
            precompute_freqs_cis(HEAD_DIM, max_seq_len, ROPE_THETA),
            persistent=False,
        )

        self._init_weights()

    def _init_weights(self):
        """GPT-style weight init: N(0, 0.02), residual projections scaled by 1/sqrt(2*n_layers)."""
        std = 0.02
        res_std = std / math.sqrt(2 * self.n_layers)
        for name, module in self.named_modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=std)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=std)
        # Scale residual projections (o_proj, W2) by 1/sqrt(2L)
        for layer in self.layers:
            nn.init.normal_(layer.attn.o_proj.weight, mean=0.0, std=res_std)
            ffn = layer.ffn.swiglu if isinstance(layer.ffn, ResonantFFN) else layer.ffn
            nn.init.normal_(ffn.W2.weight, mean=0.0, std=res_std)

    def forward(
        self,
        input_ids:      torch.Tensor,          # (B, T)
        labels:         Optional[torch.Tensor] = None,  # (B, T)
    ) -> Dict[str, torch.Tensor]:
        B, T = input_ids.shape

        h = self.embed_drop(self.embed_tokens(input_ids))  # (B, T, d_model)

        # Experience stream: zeros initialisation per sequence
        experience = None
        if self.enable_experience:
            experience = torch.zeros(B, T, D_EXP, device=input_ids.device, dtype=h.dtype)

        # Collect per-layer metrics
        curiosity_total    = torch.tensor(0.0, device=h.device)
        all_anchor_scores  = []   # list of (B, T, n_anchors) per layer
        alpha_values       = {}   # layer_idx -> alpha scalar

        for layer in self.layers:
            h, experience, curiosity, anchor_scores = layer(h, self.freqs_cis, experience)

            if curiosity is not None:
                curiosity_total = curiosity_total + curiosity.mean()

            if anchor_scores is not None:
                all_anchor_scores.append(anchor_scores)

            if layer.is_resonant:
                alpha_values[layer.layer_idx] = layer.ffn.alpha.item()

        h     = self.norm(h)
        logits = self.lm_head(h)                         # (B, T, vocab_size)

        # Loss computation
        loss    = None
        ce_loss = None
        curiosity_loss = None

        if labels is not None:
            # Shift for next-token prediction
            shift_logits = logits[:, :-1, :].contiguous().view(-1, logits.size(-1))
            shift_labels = labels[:, 1:].contiguous().view(-1)
            ce_loss = F.cross_entropy(shift_logits, shift_labels, ignore_index=-100)

            curiosity_loss = CURIOSITY_WEIGHT * curiosity_total / self.n_layers
            loss           = ce_loss + curiosity_loss

        return {
            "loss":            loss,
            "ce_loss":         ce_loss,
            "curiosity_loss":  curiosity_loss,
            "logits":          logits,
            "anchor_scores":   all_anchor_scores if all_anchor_scores else None,
            "alpha_values":    alpha_values,
            "experience_state": experience,
        }

    def generate(
        self,
        input_ids:    torch.Tensor,
        max_new_tokens: int = 200,
        temperature:  float = 1.0,
        top_k:        int   = 50,
    ) -> torch.Tensor:
        """Greedy / top-k sampling for quick eval."""
        self.eval()
        with torch.no_grad():
            for _ in range(max_new_tokens):
                # Crop to max_seq_len context
                ctx = input_ids[:, -MAX_SEQ_LEN:]
                out = self(ctx)
                logits = out["logits"][:, -1, :] / temperature
                if top_k > 0:
                    v, _ = torch.topk(logits, top_k)
                    logits[logits < v[:, [-1]]] = float('-inf')
                probs = F.softmax(logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
                input_ids = torch.cat([input_ids, next_token], dim=1)
        return input_ids

In [ ]:
# ============================================================
# PARAMETER COUNTER & ARCHITECTURE SUMMARY
# ============================================================

def count_params(model: nn.Module) -> Dict[str, int]:
    """Count parameters broken down by component group."""
    counts = defaultdict(int)
    for name, p in model.named_parameters():
        n = p.numel()
        if "embed_tokens" in name:
            counts["embeddings"] += n
        elif "q_proj" in name or "k_proj" in name or "v_proj" in name:
            counts["attention_qkv"] += n
        elif "o_proj" in name:
            counts["attention_out"] += n
        elif "anchors" in name or "W_anchor" in name or "U_k" in name or "U_v" in name:
            counts["slip_anchors"] += n
        elif "W_s" in name or "W_pred" in name or "W_e" in name:
            counts["experience_stream"] += n
        elif "W_freq" in name or "alpha_raw" in name:
            counts["resonant_ffn_extra"] += n
        elif "W1" in name or "W2" in name or "W3" in name:
            counts["swiglu_ffn"] += n
        elif "norm" in name:
            counts["layer_norms"] += n
        elif "lm_head" in name:
            counts["lm_head (tied)"] += n  # tied — don't double count
        else:
            counts["other"] += n
    return dict(counts)


# Instantiate
logger.info("Instantiating Eve-3-SABER-1B...")
model = SABER1B()

if GRADIENT_CHECKPOINTING:
    model.gradient_checkpointing_enable()  # available if you subclass PreTrainedModel

# Count params
param_breakdown = count_params(model)

# Total (excluding tied lm_head)
total_params = sum(
    p.numel()
    for n, p in model.named_parameters()
    if "lm_head" not in n
)
total_params_with_tie = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\n" + "=" * 58)
print("  Eve-3-SABER-1B  —  Parameter Count")
print("=" * 58)
for k, v in sorted(param_breakdown.items(), key=lambda x: -x[1]):
    print(f"  {k:<30s}  {v / 1e6:>8.2f} M")
print("-" * 58)
print(f"  Total (unique, no tie)         {total_params / 1e6:>8.2f} M")
print(f"  Trainable                      {trainable / 1e6:>8.2f} M")
print("=" * 58)

# Warn if far from 1B
if abs(total_params - 1e9) / 1e9 > 0.05:
    print(f"\n⚠  Param count ({total_params/1e9:.3f}B) deviates >5% from 1B target.")
    print(f"   Adjust D_FF (currently {D_FF}) to tune.")
else:
    print(f"\n✓ Param count within 5% of 1B target.")

# Memory estimate
static_gb = (total_params * 18) / 1024**3   # 18 bytes/param (BF16+AdamW)
print(f"\n  Estimated static VRAM (BF16+AdamW):  {static_gb:.1f} GB")
print(f"  With gradient checkpointing (~+0.9 GB acts):  {static_gb + 0.9:.1f} GB")

---
## Cell 5 — Tokenizer Setup

In [ ]:
# ============================================================
# TOKENIZER SETUP
# Using GPT-2 BPE tokenizer (vocab 50,257) to match architecture.
# ============================================================

logger.info("Loading GPT-2 tokenizer...")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# GPT-2 has no pad token by default — set to eos
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id

# Sanity checks
assert tokenizer.vocab_size == VOCAB_SIZE, (
    f"Tokenizer vocab {tokenizer.vocab_size} != config {VOCAB_SIZE}"
)

# Verify on a sample
_sample = "The Eve-3-SABER-1B model uses slip-anchors to modulate attention."
_tokens = tokenizer.encode(_sample)
print(f"Vocab size:   {tokenizer.vocab_size}")
print(f"EOS token:    {tokenizer.eos_token!r}  (id={tokenizer.eos_token_id})")
print(f"PAD token:    {tokenizer.pad_token!r}   (id={tokenizer.pad_token_id})")
print(f"Sample text:  {_sample!r}")
print(f"Tokenised:    {_tokens}")
print(f"Decoded back: {tokenizer.decode(_tokens)!r}")
print("\n✓ Tokenizer ready.")

---
## Cell 6 — Dataset Loading & Sequence Packing

All datasets loaded with `streaming=True` — no full download required.  
Sequences are **packed** (concatenate then split at SEQ_LEN) rather than truncated/padded.  
This ensures every token in every batch is a real training signal with no wasted positions.

In [ ]:
# ============================================================
# TOKENISE HELPER
# ============================================================

EOS_ID = tokenizer.eos_token_id

def tokenise_example(example: Dict[str, Any]) -> Optional[List[int]]:
    """
    Extract text from various dataset schemas and tokenise.
    Returns list of token ids (no padding), terminated with EOS.
    """
    # Different datasets use different field names
    text = (
        example.get("text")      or
        example.get("content")   or
        example.get("passage")   or
        example.get("document")  or ""
    )
    if not isinstance(text, str) or len(text.strip()) < 50:
        return None
    # Tokenise without special tokens; we'll add EOS as document separator
    ids = tokenizer.encode(text, add_special_tokens=False, truncation=False)
    return ids + [EOS_ID]  # append EOS as document boundary


# ============================================================
# SEQUENCE PACKING ITERABLE DATASET
# ============================================================

class PackedDataset(IterableDataset):
    """
    Wraps a HuggingFace streaming dataset and packs tokenised text
    into fixed-length chunks of size SEQ_LEN.

    Avoids any padding: documents are concatenated into a rolling
    buffer and emitted as SEQ_LEN-length slices.
    """

    def __init__(
        self,
        hf_datasets_weighted: List[Tuple],  # [(hf_dataset, weight), ...]
        seq_len:    int  = SEQ_LEN,
        seed:       int  = SEED,
        max_tokens: Optional[int] = None,
    ):
        self.datasets    = hf_datasets_weighted
        self.seq_len     = seq_len
        self.seed        = seed
        self.max_tokens  = max_tokens

    def _interleave_streams(self):
        """Yield examples from datasets weighted by their mix fractions."""
        rng = random.Random(self.seed)
        iterators = [iter(ds) for ds, _ in self.datasets]
        weights   = [w for _, w in self.datasets]

        while True:
            # Sample which dataset to draw from
            ds_idx = rng.choices(range(len(self.datasets)), weights=weights, k=1)[0]
            try:
                yield next(iterators[ds_idx])
            except StopIteration:
                # Restart exhausted stream (datasets may repeat; streaming is infinite-ish)
                iterators[ds_idx] = iter(self.datasets[ds_idx][0])
                yield next(iterators[ds_idx])

    def __iter__(self):
        buffer: List[int] = []
        tokens_emitted = 0

        for example in self._interleave_streams():
            if self.max_tokens and tokens_emitted >= self.max_tokens:
                break

            ids = tokenise_example(example)
            if ids is None:
                continue
            buffer.extend(ids)

            while len(buffer) >= self.seq_len:
                chunk = buffer[:self.seq_len]
                buffer = buffer[self.seq_len:]

                input_ids = torch.tensor(chunk, dtype=torch.long)
                # Labels == input_ids (causal LM; -100 would mask padding, but we have none)
                labels    = input_ids.clone()

                yield {"input_ids": input_ids, "labels": labels}
                tokens_emitted += self.seq_len


# ============================================================
# BUILD STREAMING DATASETS
# ============================================================

def build_streaming_datasets(data_mix: List[Tuple], tag: str = "train") -> List[Tuple]:
    """
    Load each dataset in the mix with streaming=True.
    Returns list of (hf_dataset, weight) tuples.
    """
    loaded = []
    total_weight = sum(w for _, _, w, _ in data_mix)

    for ds_id, config, weight, streaming in data_mix:
        try:
            logger.info(f"Loading {ds_id}/{config}  (weight={weight/total_weight:.2%})...")
            ds = load_dataset(
                ds_id,
                config,
                split=tag,
                streaming=streaming,
                trust_remote_code=True,
            )
            # Shuffle the stream with a large buffer (1 hour of data)
            ds = ds.shuffle(seed=SEED, buffer_size=10_000)
            loaded.append((ds, weight / total_weight))
            logger.info(f"  ✓ {ds_id}/{config} ready")
        except Exception as e:
            logger.warning(f"  ✗ Failed to load {ds_id}/{config}: {e}")
            logger.warning(f"    Skipping this source. Adjust weights if needed.")

    # Re-normalise weights after any failures
    total_w = sum(w for _, w in loaded)
    loaded  = [(ds, w / total_w) for ds, w in loaded]

    print(f"\n── {tag.upper()} Data Mix ──")
    for i, (ds, w) in enumerate(loaded):
        print(f"  {i}: {w:.2%}  →  {getattr(ds, 'info', {}).get('dataset_name', '?')}")

    return loaded


# Load pretraining datasets
logger.info("Building pretraining dataset streams...")
pretrain_datasets = build_streaming_datasets(DATA_MIX_PRETRAIN, tag="train")

pretrain_ds = PackedDataset(
    hf_datasets_weighted=pretrain_datasets,
    seq_len=SEQ_LEN,
    seed=SEED,
    max_tokens=int(TOTAL_TOKENS * ANNEAL_START_FRACTION),
)

pretrain_loader = DataLoader(
    pretrain_ds,
    batch_size=MICRO_BATCH,
    num_workers=4,
    pin_memory=True,
    prefetch_factor=2,
)

logger.info("Pretraining dataloader ready.")
logger.info(f"Sequence length: {SEQ_LEN}, Micro-batch: {MICRO_BATCH}")

---
## Cell 7 — Training Utilities

WandB logging, LR schedule, checkpoint save/load, and custom metrics.

In [ ]:
# ============================================================
# WANDB SETUP
# ============================================================

def init_wandb(resume_run_id: Optional[str] = None):
    """Initialise WandB. Pass resume_run_id to continue a run."""
    run = wandb.init(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        name=WANDB_RUN_NAME if not resume_run_id else None,
        id=resume_run_id,
        resume="allow" if resume_run_id else None,
        config={
            # Model
            "d_model":     D_MODEL,
            "n_layers":    N_LAYERS,
            "n_heads":     N_HEADS,
            "d_ff":        D_FF,
            "d_exp":       D_EXP,
            "n_anchors":   N_ANCHORS,
            "d_anchor":    D_ANCHOR,
            # Ablation flags
            "enable_anchors":    ENABLE_ANCHORS,
            "enable_experience": ENABLE_EXPERIENCE,
            "enable_resonant":   ENABLE_RESONANT,
            "predictability_mode": PREDICTABILITY_MODE,
            # Training
            "total_tokens":   TOTAL_TOKENS,
            "seq_len":        SEQ_LEN,
            "micro_batch":    MICRO_BATCH,
            "grad_accum":     GRAD_ACCUM,
            "effective_batch_tokens": _actual_batch_tokens,
            "lr":             LR,
            "lr_min":         LR_MIN,
            "warmup_steps":   WARMUP_STEPS,
            "weight_decay":   WEIGHT_DECAY,
            "grad_clip":      GRAD_CLIP,
            "dtype":          DTYPE_STR,
            "curiosity_weight": CURIOSITY_WEIGHT,
            # Hardware
            "gpu_tier":  GPU_TIER,
            "n_gpus":    N_GPUS,
        },
    )
    logger.info(f"WandB run: {run.url}")
    return run


# ============================================================
# LEARNING RATE SCHEDULE
# Cosine decay with linear warmup, floor at LR_MIN.
# ============================================================

def get_lr(step: int) -> float:
    """Compute LR at given step (cosine with linear warmup)."""
    if step < WARMUP_STEPS:
        return LR * step / max(1, WARMUP_STEPS)
    # Cosine decay from WARMUP_STEPS to TOTAL_STEPS
    progress = (step - WARMUP_STEPS) / max(1, TOTAL_STEPS - WARMUP_STEPS)
    cosine   = 0.5 * (1.0 + math.cos(math.pi * progress))
    return LR_MIN + (LR - LR_MIN) * cosine


def set_lr(optimizer: torch.optim.Optimizer, lr: float):
    for group in optimizer.param_groups:
        group["lr"] = lr


# ============================================================
# CUSTOM METRICS
# ============================================================

def compute_anchor_entropy(anchor_scores_list: List[torch.Tensor]) -> float:
    """
    Gate openness metric: mean entropy of anchor score distributions.
    High entropy → spread across many anchors (diverse routing).
    Low entropy  → collapsed to a few anchors (potential mode collapse).
    """
    if not anchor_scores_list:
        return 0.0
    entropies = []
    for scores in anchor_scores_list:  # each (B, T, n_anchors)
        # Entropy per token: -sum(p log p)
        ent = -(scores * (scores + 1e-9).log()).sum(-1)  # (B, T)
        entropies.append(ent.mean().item())
    return float(np.mean(entropies))


def compute_experience_magnitude(exp_state: Optional[torch.Tensor]) -> float:
    """Mean L2 norm of the experience stream at the final layer."""
    if exp_state is None:
        return 0.0
    return exp_state.float().norm(dim=-1).mean().item()


# ============================================================
# CHECKPOINT SAVE / LOAD
# ============================================================

def save_checkpoint(
    model:     nn.Module,
    optimizer: torch.optim.Optimizer,
    step:      int,
    tokens_seen: int,
    metrics:   Dict[str, float],
    wandb_run_id: Optional[str] = None,
):
    """
    Save full training state for bulletproof resume.
    Keeps only the last KEEP_LAST_N checkpoints.
    """
    ckpt_path = CKPT_DIR / f"step_{step:08d}"
    ckpt_path.mkdir(parents=True, exist_ok=True)

    # Save model state
    torch.save(
        model.state_dict(),
        ckpt_path / "model.pt",
    )

    # Save optimizer state
    torch.save(
        optimizer.state_dict(),
        ckpt_path / "optimizer.pt",
    )

    # Save training metadata
    meta = {
        "step":          step,
        "tokens_seen":   tokens_seen,
        "metrics":       metrics,
        "wandb_run_id":  wandb_run_id,
        "config": {
            "d_model":  D_MODEL,
            "n_layers": N_LAYERS,
            "seq_len":  SEQ_LEN,
            "enable_anchors":    ENABLE_ANCHORS,
            "enable_experience": ENABLE_EXPERIENCE,
            "enable_resonant":   ENABLE_RESONANT,
        }
    }
    with open(ckpt_path / "meta.json", "w") as f:
        json.dump(meta, f, indent=2)

    # Update "latest" pointer
    with open(CKPT_DIR / "latest.txt", "w") as f:
        f.write(str(ckpt_path))

    logger.info(f"[Step {step}] Checkpoint saved → {ckpt_path}")

    # Prune old local checkpoints
    all_ckpts = sorted(CKPT_DIR.glob("step_*"), key=lambda p: int(p.name.split("_")[1]))
    for old in all_ckpts[:-KEEP_LAST_N]:
        shutil.rmtree(old)
        logger.info(f"  Pruned old checkpoint: {old.name}")

    # ── Google Drive rolling backup ──────────────────────────
    # Copies last GDRIVE_KEEP_LAST_N checkpoints to Drive.
    # This is your insurance against Colab runtime crashes.
    if GDRIVE_BACKUP_DIR is not None:
        try:
            gdrive_ckpt = GDRIVE_BACKUP_DIR / f"step_{step:08d}"
            if gdrive_ckpt.exists():
                shutil.rmtree(gdrive_ckpt)
            shutil.copytree(ckpt_path, gdrive_ckpt)
            logger.info(f"  ✓ Backed up to Drive: {gdrive_ckpt}")

            # Prune old Drive checkpoints (keep last N)
            drive_ckpts = sorted(
                GDRIVE_BACKUP_DIR.glob("step_*"),
                key=lambda p: int(p.name.split("_")[1])
            )
            for old_drive in drive_ckpts[:-GDRIVE_KEEP_LAST_N]:
                shutil.rmtree(old_drive)
                logger.info(f"  Pruned old Drive checkpoint: {old_drive.name}")

            # Also save latest.txt to Drive
            with open(GDRIVE_BACKUP_DIR / "latest.txt", "w") as f:
                f.write(str(gdrive_ckpt))
        except Exception as e:
            logger.warning(f"  Drive backup failed: {e}. Local checkpoint is safe.")


def load_checkpoint(
    model:     nn.Module,
    optimizer: Optional[torch.optim.Optimizer] = None,
    ckpt_path: Optional[Path] = None,
) -> Dict[str, Any]:
    """
    Load from a checkpoint directory.
    If ckpt_path is None, loads from latest.txt.
    Returns metadata dict with step, tokens_seen, etc.
    """
    if ckpt_path is None:
        latest_file = CKPT_DIR / "latest.txt"
        if not latest_file.exists():
            logger.info("No checkpoint found. Starting from scratch.")

            # Try Google Drive backup as fallback
            if GDRIVE_BACKUP_DIR is not None:
                drive_latest = GDRIVE_BACKUP_DIR / "latest.txt"
                if drive_latest.exists():
                    drive_ckpt = Path(drive_latest.read_text().strip())
                    if drive_ckpt.exists():
                        logger.info(f"Found Drive backup at {drive_ckpt}. Restoring...")
                        # Copy from Drive to local
                        local_ckpt = CKPT_DIR / drive_ckpt.name
                        if local_ckpt.exists():
                            shutil.rmtree(local_ckpt)
                        shutil.copytree(drive_ckpt, local_ckpt)
                        with open(CKPT_DIR / "latest.txt", "w") as f:
                            f.write(str(local_ckpt))
                        ckpt_path = local_ckpt
                        logger.info(f"  ✓ Restored from Drive: {local_ckpt}")
                    else:
                        return {"step": 0, "tokens_seen": 0, "wandb_run_id": None}
                else:
                    return {"step": 0, "tokens_seen": 0, "wandb_run_id": None}
            else:
                return {"step": 0, "tokens_seen": 0, "wandb_run_id": None}
            return {"step": 0, "tokens_seen": 0, "wandb_run_id": None}
        ckpt_path = Path(latest_file.read_text().strip())

    logger.info(f"Loading checkpoint from {ckpt_path}...")

    # Load model weights
    state_dict = torch.load(ckpt_path / "model.pt", map_location="cpu")
    model.load_state_dict(state_dict)
    logger.info("  ✓ Model weights loaded")

    # Load optimizer state
    if optimizer is not None and (ckpt_path / "optimizer.pt").exists():
        opt_state = torch.load(ckpt_path / "optimizer.pt", map_location="cpu")
        optimizer.load_state_dict(opt_state)
        logger.info("  ✓ Optimizer state loaded")

    # Load metadata
    with open(ckpt_path / "meta.json") as f:
        meta = json.load(f)

    logger.info(f"  Resuming from step {meta['step']:,} | tokens {meta['tokens_seen']:,}")
    return meta


print("✓ Training utilities ready.")

---
## Cell 8 — Training Loop

**Option A** (below) is the custom training loop — recommended for this architecture because it gives direct access to the curiosity loss, anchor entropy, and per-layer alpha values that the HF Trainer abstracts away.

**Option B** (HF Trainer with custom `compute_loss`) is at the bottom for teams already on a Trainer pipeline.

To select Option B: skip the Option A cell and run the Option B cell instead.

In [ ]:
# ============================================================
# OPTION A — Custom Training Loop
# Supports multi-GPU via accelerate, bf16, gradient checkpointing,
# and bulletproof checkpoint resume.
# ============================================================

def train(
    model:       nn.Module,
    dataloader:  DataLoader,
    resume:      bool = True,
):
    """
    Main pretraining loop.

    Args:
        model:      Instantiated SABER1B
        dataloader: Packed sequence dataloader
        resume:     If True, attempt to resume from latest checkpoint
    """

    # ── Accelerator (handles multi-GPU / mixed precision) ──
    accelerator = Accelerator(
        mixed_precision=DTYPE_STR,         # "bf16"
        gradient_accumulation_steps=GRAD_ACCUM,
        log_with=None,                     # we handle WandB manually
    )
    device = accelerator.device
    logger.info(f"Accelerator: {accelerator.num_processes} processes, device={device}")

    # ── Optimizer ──
    # Separate weight decay groups: no decay on 1-D params (norms, biases)
    decay_params     = [p for n, p in model.named_parameters() if p.dim() >= 2]
    no_decay_params  = [p for n, p in model.named_parameters() if p.dim() < 2]
    optimizer = torch.optim.AdamW(
        [
            {"params": decay_params,    "weight_decay": WEIGHT_DECAY},
            {"params": no_decay_params, "weight_decay": 0.0},
        ],
        lr=LR,
        betas=BETAS,
        fused=True if device.type == "cuda" else False,
    )

    # ── Resume from checkpoint ──
    start_step   = 0
    tokens_seen  = 0
    wandb_run_id = None

    if resume:
        meta = load_checkpoint(model, optimizer)
        start_step   = meta.get("step",        0)
        tokens_seen  = meta.get("tokens_seen", 0)
        wandb_run_id = meta.get("wandb_run_id", None)

    # ── WandB ──
    wandb_run = None
    if accelerator.is_main_process:
        try:
            wandb_run = init_wandb(resume_run_id=wandb_run_id)
            wandb_run_id = wandb_run.id
        except Exception as e:
            logger.warning(f"WandB init failed: {e}. Continuing without logging.")

    # ── Prepare with accelerate ──
    model, optimizer, dataloader = accelerator.prepare(model, optimizer, dataloader)

    if GRADIENT_CHECKPOINTING:
        model.gradient_checkpointing_enable()

    # ── Skip already-seen batches on resume ──
    # (This is approximate for streaming datasets; exact resume would need
    # a deterministic dataset state saved in the checkpoint.)
    batches_to_skip = start_step * GRAD_ACCUM
    if batches_to_skip > 0:
        logger.info(f"Skipping {batches_to_skip:,} batches to resume at step {start_step:,}...")

    # ── Training loop ──
    model.train()
    optimizer.zero_grad()

    step      = start_step
    micro_idx = 0

    # Rolling loss accumulators for LOG_EVERY window
    log_ce_loss    = 0.0
    log_cur_loss   = 0.0
    log_count      = 0
    log_anchor_ent = 0.0
    log_exp_mag    = 0.0
    t0 = time.time()

    pbar = tqdm(
        total=TOTAL_STEPS,
        initial=start_step,
        desc="Training",
        disable=not accelerator.is_main_process,
        dynamic_ncols=True,
    )

    for batch in dataloader:
        if step >= TOTAL_STEPS:
            break

        input_ids = batch["input_ids"].to(device)
        labels    = batch["labels"].to(device)

        # ── Forward pass (inside gradient accumulation context) ──
        with accelerator.accumulate(model):
            outputs = model(input_ids=input_ids, labels=labels)

            loss          = outputs["loss"]
            ce_loss       = outputs["ce_loss"]
            curiosity_loss = outputs["curiosity_loss"]

            # Accumulate metrics for logging
            log_ce_loss  += ce_loss.item()  if ce_loss  is not None else 0.0
            log_cur_loss += curiosity_loss.item() if curiosity_loss is not None else 0.0

            if outputs["anchor_scores"] is not None:
                log_anchor_ent += compute_anchor_entropy(outputs["anchor_scores"])

            if outputs["experience_state"] is not None:
                log_exp_mag += compute_experience_magnitude(outputs["experience_state"])

            log_count += 1

            # Backward
            accelerator.backward(loss)

        micro_idx += 1

        # ── Optimizer step (every GRAD_ACCUM micro-batches) ──
        if micro_idx % GRAD_ACCUM == 0:
            # Gradient clipping
            accelerator.clip_grad_norm_(model.parameters(), GRAD_CLIP)

            # Update LR
            lr = get_lr(step)
            set_lr(optimizer, lr)

            optimizer.step()
            optimizer.zero_grad()

            step        += 1
            tokens_seen += _actual_batch_tokens

            # ── Logging ──
            if step % LOG_EVERY == 0 and accelerator.is_main_process:
                elapsed = time.time() - t0
                tok_per_s = (LOG_EVERY * _actual_batch_tokens) / elapsed

                avg_ce   = log_ce_loss  / log_count
                avg_cur  = log_cur_loss / log_count
                avg_ent  = log_anchor_ent / log_count
                avg_mag  = log_exp_mag / log_count

                # Collect per-layer alpha values
                alpha_dict = {}
                raw_model = accelerator.unwrap_model(model)
                for layer in raw_model.layers:
                    if layer.is_resonant:
                        alpha_dict[f"alpha/layer_{layer.layer_idx}"] = layer.ffn.alpha.item()

                log_dict = {
                    "train/loss":            avg_ce + avg_cur,
                    "train/ce_loss":         avg_ce,
                    "train/curiosity_loss":  avg_cur,
                    "train/lr":              lr,
                    "train/tokens_seen":     tokens_seen,
                    "train/tok_per_sec":     tok_per_s,
                    "train/ppl":             math.exp(min(avg_ce, 20)),  # cap to avoid overflow
                    "metrics/anchor_entropy":    avg_ent,
                    "metrics/experience_mag":    avg_mag,
                    **alpha_dict,
                }

                pbar.set_postfix({
                    "loss":  f"{avg_ce:.4f}",
                    "ppl":   f"{math.exp(min(avg_ce, 20)):.2f}",
                    "lr":    f"{lr:.2e}",
                    "tok/s": f"{tok_per_s:.0f}",
                })

                if wandb_run:
                    wandb.log(log_dict, step=step)

                # Reset accumulators
                log_ce_loss = log_cur_loss = log_anchor_ent = log_exp_mag = 0.0
                log_count = 0
                t0 = time.time()

            # ── Checkpointing ──
            if step % CHECKPOINT_EVERY == 0 and accelerator.is_main_process:
                raw_model = accelerator.unwrap_model(model)
                save_checkpoint(
                    model=raw_model,
                    optimizer=optimizer,
                    step=step,
                    tokens_seen=tokens_seen,
                    metrics={"ce_loss": log_ce_loss / max(1, log_count)},
                    wandb_run_id=wandb_run_id,
                )

            pbar.update(1)

    pbar.close()

    # Final checkpoint
    if accelerator.is_main_process:
        raw_model = accelerator.unwrap_model(model)
        save_checkpoint(
            model=raw_model,
            optimizer=optimizer,
            step=step,
            tokens_seen=tokens_seen,
            metrics={},
            wandb_run_id=wandb_run_id,
        )
        if wandb_run:
            wandb.finish()

    logger.info(f"Training complete. Steps: {step:,} | Tokens: {tokens_seen:,}")
    return model


# ── Run training ──
# Uncomment to start training:
#
# model = train(
#     model=model,
#     dataloader=pretrain_loader,
#     resume=True,
# )

In [ ]:
# ============================================================
# OPTION B — HuggingFace Trainer with custom compute_loss
# Use this if your team already uses the Trainer API.
# NOTE: Requires wrapping SABER1B in a PreTrainedModel shell.
# ============================================================

from transformers import Trainer, TrainingArguments, PreTrainedModel, PretrainedConfig


class SABERConfig(PretrainedConfig):
    model_type = "saber"

    def __init__(
        self,
        d_model: int = D_MODEL,
        n_layers: int = N_LAYERS,
        n_heads: int = N_HEADS,
        d_ff: int = D_FF,
        vocab_size: int = VOCAB_SIZE,
        max_seq_len: int = MAX_SEQ_LEN,
        d_exp: int = D_EXP,
        n_anchors: int = N_ANCHORS,
        d_anchor: int = D_ANCHOR,
        enable_anchors: bool = ENABLE_ANCHORS,
        enable_experience: bool = ENABLE_EXPERIENCE,
        enable_resonant: bool = ENABLE_RESONANT,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.d_model          = d_model
        self.n_layers         = n_layers
        self.n_heads          = n_heads
        self.d_ff             = d_ff
        self.vocab_size       = vocab_size
        self.max_seq_len      = max_seq_len
        self.d_exp            = d_exp
        self.n_anchors        = n_anchors
        self.d_anchor         = d_anchor
        self.enable_anchors   = enable_anchors
        self.enable_experience = enable_experience
        self.enable_resonant  = enable_resonant


class SABER1BForCausalLM(PreTrainedModel):
    """Thin PreTrainedModel wrapper for HF Trainer compatibility."""
    config_class = SABERConfig

    def __init__(self, config: SABERConfig):
        super().__init__(config)
        self.model = SABER1B(
            vocab_size=config.vocab_size,
            d_model=config.d_model,
            n_layers=config.n_layers,
            max_seq_len=config.max_seq_len,
            enable_anchors=config.enable_anchors,
            enable_experience=config.enable_experience,
            enable_resonant=config.enable_resonant,
        )

    def forward(self, input_ids, labels=None, **kwargs):
        out = self.model(input_ids=input_ids, labels=labels)
        # HF Trainer expects a loss key and logits key
        return type("Output", (), {
            "loss":   out["loss"],
            "logits": out["logits"],
        })()


class SABERTrainer(Trainer):
    """Custom Trainer that logs curiosity loss, anchor entropy, and alphas."""

    def compute_loss(self, model, inputs, return_outputs=False):
        outputs = model.model(
            input_ids=inputs["input_ids"],
            labels=inputs["labels"],
        )
        loss = outputs["loss"]

        # Log component losses to W&B if available
        if self.args.report_to and "wandb" in self.args.report_to:
            wandb.log({
                "train/ce_loss":        outputs["ce_loss"].item() if outputs["ce_loss"] is not None else 0,
                "train/curiosity_loss": outputs["curiosity_loss"].item() if outputs["curiosity_loss"] is not None else 0,
                "metrics/anchor_entropy": compute_anchor_entropy(
                    outputs["anchor_scores"] or []
                ),
                "metrics/experience_mag": compute_experience_magnitude(
                    outputs["experience_state"]
                ),
                **{f"alpha/layer_{k}": v for k, v in outputs["alpha_values"].items()},
            }, commit=False)

        return (loss, outputs) if return_outputs else loss


# Option B training config
training_args_b = TrainingArguments(
    output_dir=str(CKPT_DIR),
    max_steps=TOTAL_STEPS,
    per_device_train_batch_size=MICRO_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    adam_beta1=BETAS[0],
    adam_beta2=BETAS[1],
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type="cosine",
    bf16=True,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
    max_grad_norm=GRAD_CLIP,
    logging_steps=LOG_EVERY,
    save_steps=CHECKPOINT_EVERY,
    save_total_limit=KEEP_LAST_N,
    report_to=["wandb"],
    run_name=WANDB_RUN_NAME,
    dataloader_num_workers=4,
    remove_unused_columns=False,
)

# ── To use Option B, uncomment: ──
#
# config_b = SABERConfig()
# model_b  = SABER1BForCausalLM(config_b)
# trainer  = SABERTrainer(
#     model=model_b,
#     args=training_args_b,
#     train_dataset=pretrain_ds,
#     tokenizer=tokenizer,
# )
# trainer.train(resume_from_checkpoint=True)

print("✓ Option B (HF Trainer) configured.")

---
## Cell 9 — Evaluation

Quick eval after training (or mid-training): sample generations, perplexity on held-out data,
and detailed component analysis (alpha values, anchor usage distribution).

In [ ]:
# ============================================================
# EVALUATION SUITE
# ============================================================

@torch.no_grad()
def evaluate_perplexity(
    model:    nn.Module,
    n_tokens: int = 1_000_000,
    tag:      str = "eval",
) -> float:
    """
    Estimate perplexity on held-out FineWeb-Edu samples.
    Uses streaming so no downloads needed.
    """
    model.eval()
    device = next(model.parameters()).device

    logger.info(f"Evaluating perplexity on ~{n_tokens/1e6:.1f}M tokens...")

    eval_ds_raw = load_dataset(
        "HuggingFaceFW/fineweb-edu",
        "sample-10BT",
        split="train",
        streaming=True,
    ).skip(50_000)  # skip a block that was NOT in train mix

    eval_packed = PackedDataset(
        hf_datasets_weighted=[(eval_ds_raw, 1.0)],
        seq_len=SEQ_LEN,
        seed=999,
        max_tokens=n_tokens,
    )
    eval_loader = DataLoader(eval_packed, batch_size=MICRO_BATCH, num_workers=2)

    total_ce   = 0.0
    total_toks = 0

    for batch in tqdm(eval_loader, desc="Eval", leave=False):
        ids    = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        out    = model(input_ids=ids, labels=labels)
        total_ce   += out["ce_loss"].item() * ids.numel()
        total_toks += ids.numel()

    mean_ce = total_ce / total_toks
    ppl     = math.exp(mean_ce)
    logger.info(f"[{tag}] Perplexity: {ppl:.2f}  (CE loss: {mean_ce:.4f})")
    model.train()
    return ppl


@torch.no_grad()
def sample_generations(
    model:     nn.Module,
    prompts:   List[str] = None,
    max_new:   int = 200,
    temperature: float = 0.8,
):
    """Generate samples for qualitative eval."""
    model.eval()
    device = next(model.parameters()).device

    if prompts is None:
        prompts = [
            "The history of machine learning begins with",
            "In mathematics, a prime number is",
            "def fibonacci(n):\n    ",
            "The SABER model architecture introduces a novel",
        ]

    print("\n" + "=" * 60)
    print("  Sample Generations")
    print("=" * 60)

    for prompt in prompts:
        ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
        gen = model.generate(ids, max_new_tokens=max_new, temperature=temperature)
        text = tokenizer.decode(gen[0], skip_special_tokens=True)
        print(f"\nPrompt: {prompt!r}")
        print(f"Output: {text}")
        print("-" * 60)

    model.train()


@torch.no_grad()
def component_analysis(model: nn.Module):
    """Print current alpha values, anchor distribution, experience norm stats."""
    model.eval()
    device = next(model.parameters()).device

    print("\n" + "=" * 60)
    print("  Component Analysis")
    print("=" * 60)

    # ── Alpha values (Resonant FFN) ──
    if ENABLE_RESONANT:
        print("\n  Resonant FFN — alpha (sigmoid of alpha_raw):")
        for layer in model.layers:
            if layer.is_resonant:
                a = layer.ffn.alpha.item()
                bar = "█" * int(a * 20)
                print(f"    Layer {layer.layer_idx:2d}: {a:.4f}  {bar}")

    # ── Anchor codebook analysis ──
    if ENABLE_ANCHORS:
        print("\n  Slip-Anchors — codebook L2 norms (first 4 layers):")
        for layer in model.layers[:4]:
            if hasattr(layer.attn, 'anchors'):
                norms = layer.attn.anchors.anchors.weight.norm(dim=-1)  # (n_anchors,)
                print(f"    Layer {layer.layer_idx}: min={norms.min():.3f}  max={norms.max():.3f}  mean={norms.mean():.3f}")

    # ── Run a single forward pass to get live anchor scores ──
    print("\n  Live anchor entropy on test prompt:")
    test_ids = tokenizer.encode(
        "The transformer architecture has revolutionised natural language processing.",
        return_tensors="pt"
    ).to(device)

    out = model(input_ids=test_ids)

    if out["anchor_scores"] is not None:
        for i, scores in enumerate(out["anchor_scores"]):
            ent = -(scores * (scores + 1e-9).log()).sum(-1).mean().item()
            print(f"    Layer {i}: entropy={ent:.4f}")

    if out["experience_state"] is not None:
        exp_mag = out["experience_state"].norm(dim=-1).mean().item()
        print(f"\n  Experience stream final-layer magnitude: {exp_mag:.4f}")

    model.train()


# ── Run evaluations ──
# After training completes (or resume from checkpoint), run:
#
# load_checkpoint(model)   # load best checkpoint if needed
# ppl = evaluate_perplexity(model, n_tokens=5_000_000)
# sample_generations(model)
# component_analysis(model)

print("✓ Evaluation functions ready.")

---
## Cell 10 — Save & Push to HuggingFace Hub

In [ ]:
# ============================================================
# SAVE MODEL — SAFETENSORS FORMAT
# ============================================================

from safetensors.torch import save_file as safe_save
from huggingface_hub import HfApi, create_repo, upload_folder


def save_model_for_hf(
    model:      nn.Module,
    save_dir:   Path = MODEL_DIR,
):
    """
    Save model weights in safetensors format + config + tokenizer.
    Produces a directory ready for `push_to_hub`.
    """
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    # ── Save weights in safetensors ──
    logger.info("Saving model weights (safetensors)...")
    state_dict = {k: v.contiguous().cpu() for k, v in model.state_dict().items()}
    safe_save(state_dict, str(save_dir / "model.safetensors"))
    logger.info(f"  ✓ Saved to {save_dir / 'model.safetensors'}")

    # ── Save config ──
    config = {
        "model_type":         "saber",
        "architectures":      ["SABER1BForCausalLM"],
        "auto_map": {
            "AutoConfig":          "configuration_saber.SABERConfig",
            "AutoModelForCausalLM": "modeling_saber.SABER1BForCausalLM",
        },
        # Architecture dims
        "d_model":            D_MODEL,
        "n_layers":           N_LAYERS,
        "n_heads":            N_HEADS,
        "head_dim":           HEAD_DIM,
        "d_ff":               D_FF,
        "vocab_size":         VOCAB_SIZE,
        "max_seq_len":        MAX_SEQ_LEN,
        "rope_theta":         ROPE_THETA,
        # Novel components
        "d_exp":              D_EXP,
        "n_anchors":          N_ANCHORS,
        "d_anchor":           D_ANCHOR,
        "enable_anchors":     ENABLE_ANCHORS,
        "enable_experience":  ENABLE_EXPERIENCE,
        "enable_resonant":    ENABLE_RESONANT,
        # Training metadata
        "total_tokens_trained": TOTAL_TOKENS,
        "training_lr":        LR,
        "training_dtype":     DTYPE_STR,
    }

    with open(save_dir / "config.json", "w") as f:
        json.dump(config, f, indent=2)
    logger.info(f"  ✓ Saved config.json")

    # ── Save tokenizer ──
    tokenizer.save_pretrained(str(save_dir))
    logger.info(f"  ✓ Saved tokenizer")

    # ── Generation config ──
    gen_config = {
        "bos_token_id":  tokenizer.bos_token_id,
        "eos_token_id":  tokenizer.eos_token_id,
        "pad_token_id":  tokenizer.pad_token_id,
        "max_new_tokens": 512,
        "do_sample":     True,
        "temperature":   0.8,
        "top_p":         0.9,
    }
    with open(save_dir / "generation_config.json", "w") as f:
        json.dump(gen_config, f, indent=2)
    logger.info(f"  ✓ Saved generation_config.json")

    logger.info(f"\n✓ Model saved to {save_dir}")
    return save_dir


# ============================================================
# MODEL CARD GENERATION
# ============================================================

def generate_model_card(save_dir: Path):
    card = f"""---
language: en
license: apache-2.0
tags:
  - causal-lm
  - saber
  - slip-anchors
  - experience-stream
  - resonant-ffn
  - pretrained
base_model: null
---

# Eve-3-SABER-1B

A 1B-parameter dense decoder-only language model pretrained from scratch on 100B tokens with three novel architectural components.

## Architecture

| Component | Value |
|-----------|-------|
| Parameters | ~1.0B |
| Layers | 24 |
| d_model | 2048 |
| Attention heads | 16 (head_dim=128) |
| FFN dim | 2855 (tuned) |
| Vocabulary | 50,257 (GPT-2) |
| Context length | 2048 |
| Positional encoding | RoPE (theta=10000) |

## Novel Components

### Slip-Anchors
A learnable codebook of 64 anchor vectors in 128-dimensional space. Each token scores itself against the codebook and biases K and V additively after RoPE, enabling soft semantic routing while preserving FlashAttention compatibility.

### Experience Stream
A per-token, 256-dimensional side-channel that flows layer-to-layer (not token-to-token). Each layer predicts the next layer's experience summary; the prediction error drives an auxiliary curiosity loss (weight=0.01) that encourages the stream to carry non-degenerate information.

### Resonant FFN
Even-indexed layers blend standard SwiGLU output with a sinusoidal modulation `sin(W_freq @ x)` via a learned per-layer scalar alpha (initialised to 0.95 — nearly pure SwiGLU at the start of training).

## Training

- **Tokens:** 100B  
- **Data mix:** 55% FineWeb-Edu, 35% DCLM-Baseline, 10% StarCoderData + OpenWebMath  
- **Annealing (last 10%):** DCLM, FineMath-4+, Stack-Edu, Wikipedia, Cosmopedia v2  
- **Optimizer:** AdamW (lr=3e-4, β=(0.9, 0.95), wd=0.1)  
- **Schedule:** Cosine with 2000-step linear warmup  
- **Precision:** BF16 mixed  
- **Hardware:** RunPod H100 PCIe  

## Usage

```python
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("{HF_REPO}")
model     = AutoModelForCausalLM.from_pretrained("{HF_REPO}", trust_remote_code=True)

inputs = tokenizer("The SABER model", return_tensors="pt")
output = model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(output[0]))
```

## License
Apache 2.0
"""
    with open(save_dir / "README.md", "w") as f:
        f.write(card)
    logger.info("  ✓ Saved README.md (model card)")


# ============================================================
# PUSH TO HF HUB
# ============================================================

def push_to_hub(save_dir: Path = MODEL_DIR, repo_id: str = HF_REPO):
    """
    Push the saved model directory to HuggingFace Hub.
    Requires HF_TOKEN env variable or `huggingface-cli login`.
    """
    api = HfApi()

    # Create repo if it doesn't exist
    try:
        create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
        logger.info(f"  Repo: https://huggingface.co/{repo_id}")
    except Exception as e:
        logger.warning(f"  create_repo: {e}")

    # Upload entire folder
    logger.info(f"Uploading {save_dir} → {repo_id}...")
    api.upload_folder(
        folder_path=str(save_dir),
        repo_id=repo_id,
        repo_type="model",
        commit_message="Upload Eve-3-SABER-1B pretrained weights",
    )
    logger.info(f"✓ Pushed to https://huggingface.co/{repo_id}")


# ── Run save + push ──
# After training:
#
# save_dir = save_model_for_hf(model)
# generate_model_card(save_dir)
# push_to_hub(save_dir)

print("✓ Save & Hub upload functions ready.")

---
## Cell 11 — Annealing Phase (Optional)

Resume from the last checkpoint and switch to the high-quality annealing data mix.  
LR decays linearly from its current value to 0 over the remaining `ANNEAL_TOKENS`.

This phase mimics OLMo 2's Dolmino mid-training stage and typically yields **+10–20% benchmark improvement**.

In [ ]:
# ============================================================
# ANNEALING PHASE
# ============================================================

def run_annealing(model: nn.Module):
    """
    Annealing phase:
      - Switch to high-quality annealing data mix
      - Decay LR linearly to 0 over ANNEAL_TOKENS
      - Resume from latest checkpoint
    """

    logger.info("=" * 58)
    logger.info("  Starting ANNEALING phase")
    logger.info(f"  Anneal tokens: {ANNEAL_TOKENS:,}")
    logger.info("=" * 58)

    # ── Load annealing datasets ──
    anneal_datasets = build_streaming_datasets(DATA_MIX_ANNEAL, tag="train")

    anneal_ds = PackedDataset(
        hf_datasets_weighted=anneal_datasets,
        seq_len=SEQ_LEN,
        seed=SEED + 1,
        max_tokens=ANNEAL_TOKENS,
    )

    anneal_loader = DataLoader(
        anneal_ds,
        batch_size=MICRO_BATCH,
        num_workers=4,
        pin_memory=True,
    )

    # ── Accelerator ──
    accelerator = Accelerator(
        mixed_precision=DTYPE_STR,
        gradient_accumulation_steps=GRAD_ACCUM,
    )
    device = accelerator.device

    # ── Optimizer (fresh for annealing) ──
    decay_params    = [p for n, p in model.named_parameters() if p.dim() >= 2]
    no_decay_params = [p for n, p in model.named_parameters() if p.dim() < 2]
    optimizer = torch.optim.AdamW(
        [
            {"params": decay_params,    "weight_decay": WEIGHT_DECAY},
            {"params": no_decay_params, "weight_decay": 0.0},
        ],
        lr=LR_MIN,   # Start annealing from LR_MIN (end-of-cosine schedule)
        betas=BETAS,
    )

    # Load checkpoint (resume from pretraining end)
    meta = load_checkpoint(model, optimizer)
    start_step  = meta.get("step",        0)
    tokens_seen = meta.get("tokens_seen", 0)

    model, optimizer, anneal_loader = accelerator.prepare(model, optimizer, anneal_loader)

    anneal_steps = ANNEAL_TOKENS // _actual_batch_tokens
    logger.info(f"  Annealing steps: {anneal_steps:,}")

    # LR schedule: linear from LR_MIN → 0 over anneal_steps
    def get_anneal_lr(anneal_step: int) -> float:
        return LR_MIN * max(0.0, 1.0 - anneal_step / anneal_steps)

    # ── Annealing loop ──
    model.train()
    optimizer.zero_grad()
    micro_idx = 0
    anneal_step = 0

    pbar = tqdm(
        total=anneal_steps,
        desc="Annealing",
        disable=not accelerator.is_main_process,
    )

    for batch in anneal_loader:
        if anneal_step >= anneal_steps:
            break

        input_ids = batch["input_ids"].to(device)
        labels    = batch["labels"].to(device)

        with accelerator.accumulate(model):
            outputs = model(input_ids=input_ids, labels=labels)
            loss    = outputs["loss"]
            accelerator.backward(loss)

        micro_idx += 1

        if micro_idx % GRAD_ACCUM == 0:
            accelerator.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            lr = get_anneal_lr(anneal_step)
            set_lr(optimizer, lr)
            optimizer.step()
            optimizer.zero_grad()
            anneal_step += 1
            tokens_seen += _actual_batch_tokens

            pbar.update(1)
            pbar.set_postfix({
                "loss": f"{outputs['ce_loss'].item():.4f}",
                "lr":   f"{lr:.2e}",
            })

            # Checkpoint every CHECKPOINT_EVERY steps
            if anneal_step % CHECKPOINT_EVERY == 0 and accelerator.is_main_process:
                raw_model = accelerator.unwrap_model(model)
                save_checkpoint(
                    model=raw_model,
                    optimizer=optimizer,
                    step=start_step + anneal_step,
                    tokens_seen=tokens_seen,
                    metrics={},
                )

    pbar.close()

    # Final save
    if accelerator.is_main_process:
        raw_model = accelerator.unwrap_model(model)
        save_checkpoint(
            model=raw_model,
            optimizer=optimizer,
            step=start_step + anneal_step,
            tokens_seen=tokens_seen,
            metrics={},
        )

    logger.info("Annealing complete.")
    return accelerator.unwrap_model(model)


# ── Run annealing ──
# After pretraining is done:
#
# model = run_annealing(model)
# save_dir = save_model_for_hf(model)
# generate_model_card(save_dir)
# push_to_hub(save_dir)

print("\n✓ Annealing phase ready.")
print(f"  Annealing data mix:")
for ds_id, cfg, w, _ in DATA_MIX_ANNEAL:
    print(f"    {w*100:.0f}%  {ds_id}/{cfg}")
print(f"\n  Anneal tokens: {ANNEAL_TOKENS:,}  ({ANNEAL_TOKENS/TOTAL_TOKENS*100:.0f}% of total)")
print(f"  LR decay: {LR_MIN:.1e} → 0 (linear)")

---
## Quick-Start Checklist

1. **Run Cell 2** to install dependencies.
2. **Run Cell 3** and verify batch config in the printed output. Optionally uncomment a platform override block.
3. **Run Cells 4–5** to instantiate the model and tokenizer. Check param count is ~1B.
4. **Run Cell 6** — datasets load with streaming; no download needed.
5. **Run Cells 7–8** to define the training loop.
6. **Uncomment** the `train(...)` call in Cell 8 to start training.
7. Training checkpoints to `./checkpoints/step_XXXXXXXX/`. Resume at any time by running from Cell 3 → Cell 8 with `resume=True`.
8. After training, run **Cell 9** for eval, **Cell 10** to save and push to Hub, **Cell 11** for annealing.

---

## Notes on Robustness (Lessons from Eve-2)

- **Checkpoint every 1000 steps** (~30 min on H100 PCIe). `latest.txt` always points to the newest checkpoint.
- **WandB run ID** is saved in `meta.json` so the same run is resumed correctly on W&B dashboard.
- **Streaming datasets** are stateless — resume skips ahead by step×GRAD_ACCUM batches approximately. For exact resume you would need to save dataset state; the current skip-ahead is good enough for most use cases.
- **Curiosity stop-gradient** is non-negotiable: without `s.detach()` the experience stream will produce zero gradient signal or collapse.
- **Flash Attention** falls back gracefully to PyTorch SDPA if `flash-attn` is unavailable — no code changes needed.
- **Anchor K-bias and V-bias** share a single `SlipAnchorModule` across all heads, broadcast via `unsqueeze(2)`. This is the spec-compliant implementation and saves ~15M params vs. per-head codebooks.

---
*Notebook generated for Eve-3-SABER-1B pretraining. Architecture from consolidated model council spec. Dataset strategy from research_datasets.md. Hardware targets from research_compute.md.*